# LLM-JEPA Symbolic Regression: Inference & Evaluation
This notebook provides a high-level interface for evaluating trained LLM-JEPA models on the **AI Feynman (AIF)** dataset.

It uses the `ModelEvaluator` module to:
1.  **Run mass evaluation** for R2, recovery rate, and node complexity.
2.  **Inspect specific equations** by ID.
3.  **Generate a Prediction Gallery** in Markdown format.

---

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path for module imports
if ".." not in sys.path:
    sys.path.append("..")

import torch
from models.evaluator import ModelEvaluator
from evaluation.evaluate import print_results

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

## 1. Configuration & Checkpoint Selection
Select the model checkpoint you wish to evaluate. By default, we look for `checkpoints/last.ckpt`.

In [ ]:
# Paths relative to project root
CONFIG_PATH = "../configs/base_config.yaml"
CKPT_PATH   = "../checkpoints/last.ckpt"

# Verify paths
if not Path(CKPT_PATH).exists():
    print(f"Warning: Checkpoint not found at {CKPT_PATH}")
else:
    print(f"Found checkpoint: {CKPT_PATH}")

## 2. Initialize Evaluator
We initialize the `ModelEvaluator` which handles the pruned **InferenceModel** architecture (removing the training-only Target Encoder and Predictor).

In [ ]:
evaluator = ModelEvaluator(
    config_path=CONFIG_PATH,
    ckpt_path=CKPT_PATH,
    device=device
)

## 3. Full AI Feynman Evaluation
Run the complete evaluation suite. This will iterate through the Feynman equations and compute mass metrics.

Results are saved to `results/` for persistence.

In [ ]:
# This might take a few minutes depending on GPU speed
metrics = evaluator.run_evaluation(output_dir="../results/notebook_run", verbose=True)

# Print a formatted summary table
print_results(metrics)

## 4. Individual Equation Inspection
Test the model's performance on a specific equation from the Feynman dataset.

In [ ]:
# Select an ID (e.g., "I.6.2a", "I.8.14")
eq_id = "I.6.2a" 

sample = evaluator.predict_sample_by_id(eq_id)

if sample:
    print(f"Equation ID:  {sample['id']}")
    print(f"Ground Truth: {sample['gt']}")
    print(f"Prediction:   {sample['pred']}")
    print(f"RPN Tokens:   {' '.join(sample['tokens'])}")
else:
    print(f"Equation {eq_id} not found in dataset.")

## 5. Results Gallery
Inspecting the top-level recoveries from the evaluation run.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Collect results into a dataframe for easy viewing
results_list = []
for res in metrics['per_eq_results'][:15]: # Show first 15
    results_list.append({
        "ID": res['eq_id'],
        "Recovery": "Pass" if res['exact'] else "Fail",
        "R2 (post-BFGS)": f"{res['r2_post_bfgs']:.4f}",
        "Prediction": res['predicted'] if res['predicted'] else "Failed"
    })

df = pd.DataFrame(results_list)
display(df)